# 35장 · 대화 상태 · add_messages와 체크포인터로 이력 관리하기

add_messages는 기존 메시지와 새 메시지를 ID 기준으로 합칩니다. 새 ID면 뒤에 붙이고 기존 ID면 그 메시지를 교체합니다. thread_id는 대화 전체를 구분할 때 쓰는 별도의 값입니다.

## 메시지 ID를 유지하고 내용 바꾸기

`add_messages(기존 목록, 새 목록)`는 메시지 ID를 보고 기존 메시지를 갱신하거나 목록에 추가합니다. 아래에서는 내용만 바꾸고 두 ID를 같게 두었습니다. 새 커널에서는 `%pip install "langgraph>=1,<2"`로 준비하세요.

In [ ]:
from langchain_core.messages import HumanMessage
from langgraph.graph.message import add_messages

old = HumanMessage(content="안녕", id="1")
new = HumanMessage(content="반가워", id="1")
print([m.content for m in add_messages([old], [new])])

['반가워']


두 메시지의 ID가 같아 결과에는 “반가워” 하나만 남습니다. new의 ID를 “2”로 바꾸면 두 메시지가 모두 남습니다.

## 응용 실습 · 체크포인터로 대화 이어 가기

짝 활동에는 [실습 수첩](pre-mid/documents/graph/workshop.md)을 사용합니다. 과제 아래에서 풀이를 펼칠 수 있습니다.

## 직전 답이 지워지는 순간

상태에서 일반적인 리스트 칸에 새 리스트를 반환하면 그 칸의 값은 새 값으로 교체됩니다. 대화에서는 사용자 입력만 보내더라도 기존 발화와 새 발화를 합쳐야 합니다. MessagesState의 messages 칸은 메시지 ID를 기준으로 합치는 add_messages 규칙을 갖습니다. 새 ID면 추가하고, 이미 있던 ID면 같은 메시지를 갱신합니다.

교재 ZIP을 풀고 그 폴더에서 Jupyter를 엽니다. Python 3.12 커널에서 필요한 패키지가 없으면 새 셀에서 `%pip install "langgraph>=1,<2" openai streamlit pypdf python-dotenv`를 실행하고 커널을 다시 시작하세요. 실습 자료 폴더와 노트북은 함께 보관합니다.

`add_messages(기존메시지, 새메시지)`는 메시지 목록을 합치는 규칙입니다. 역할과 본문이 같아도 ID가 다르면 두 발화가 될 수 있고, 기존 ID와 같은 ID를 명시하면 그 자리를 갱신합니다. 아래에서 두 번째 메시지의 `id`를 보면서 “누적”과 “정정”의 목록 길이를 먼저 예상하세요.

In [1]:
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph.message import add_messages

first_turn = [HumanMessage(content="제 이름은 하늘입니다", id="first")]
next_turn = [HumanMessage(content="제 이름은 무엇인가요?", id="second")]
print("교체", [m.content for m in next_turn])
print("누적", [m.content for m in add_messages(first_turn, next_turn)])
print("정정", [m.content for m in add_messages(first_turn, [HumanMessage(content="이름은 하늘", id="first")])])

교체 ['제 이름은 무엇인가요?']
누적 ['제 이름은 하늘입니다', '제 이름은 무엇인가요?']
정정 ['이름은 하늘']


두 번째 줄은 발화 두 개가 남고 셋째 줄은 한 개만 남습니다. 학생이 매번 전체 이력을 새 메시지로 만들어 보내면 동일한 발화를 중복 입력할 수 있습니다. 이후에는 새 질문 하나만 전달합니다.

## InMemorySaver와 thread_id로 대화 이어 가기

한 번의 invoke가 끝나면 상태도 그 실행에서 끝납니다. 체크포인터는 실행 사이에 상태를 보관합니다. 이 수업의 InMemorySaver는 현재 프로세스 메모리에 두므로 프로세스를 종료하면 사라집니다. thread_id는 어떤 대화 기록을 이어 쓸지 고르는 값입니다.

`compile(checkpointer=InMemorySaver())`로 만든 그래프는 실행 사이의 상태를 메모리에 보관합니다. `invoke`의 첫 인자는 이번 발화이고 두 번째 인자는 어느 대화를 이어 갈지 정하는 설정입니다. `thread_id`가 같은 두 호출 사이에 다른 대화가 끼어도 같은 기록을 이어 읽습니다. 새 저장 객체를 만들거나 프로세스를 끝내면 이 메모리의 이전 상태를 꺼낼 수 없습니다.

In [2]:
from langgraph.graph import START, END, MessagesState, StateGraph
from langgraph.checkpoint.memory import InMemorySaver

def build_chat(reply):
    plan = StateGraph(MessagesState)
    def respond(state):
        # 기존 메시지는 상태에 남아 있으므로 새 AI 답 하나만 반환합니다.
        return {"messages": [AIMessage(content=reply(state["messages"]))]}
    plan.add_node("respond", respond)
    plan.add_edge(START, "respond")
    plan.add_edge("respond", END)
    return plan.compile(checkpointer=InMemorySaver())

def config(thread):
    return {"configurable": {"thread_id": thread}}

def send(graph, thread, text):
    return graph.invoke({"messages": [HumanMessage(content=text)]}, config(thread))

# 모델 대신 상태를 읽는 관찰용 응답입니다.
chat = build_chat(lambda messages: f"처음 받은 말: {messages[0].content}")
send(chat, "blue", "제 이름은 하늘입니다")
send(chat, "green", "제 이름은 바다입니다")
print(send(chat, "blue", "제 이름은 무엇인가요?")["messages"][-1].content)

처음 받은 말: 제 이름은 하늘입니다


관찰용 응답은 첫 사용자 메시지를 읽습니다. 파란 스레드와 초록 스레드에서 서로 다른 이름이 나옵니다. 이어지는 `chat.get_state(config(thread))`는 **저장된 상태를 조회**합니다. 새 질문을 보내는 `invoke`와 달리 응답 노드를 다시 실행하지 않습니다. `.values["messages"]`에서 그 스레드의 메시지 목록을 읽으세요.

In [3]:
for thread in ("blue", "green"):
    saved = chat.get_state(config(thread)).values["messages"]
    print(thread, [(m.type, m.content) for m in saved])

blue [('human', '제 이름은 하늘입니다'), ('ai', '처음 받은 말: 제 이름은 하늘입니다'), ('human', '제 이름은 무엇인가요?'), ('ai', '처음 받은 말: 제 이름은 하늘입니다')]
green [('human', '제 이름은 바다입니다'), ('ai', '처음 받은 말: 제 이름은 바다입니다')]


blue에는 사용자·답 두 쌍, green에는 한 쌍이 남습니다. 화면에 대화가 사라져도 체크포인트에 남을 수 있으므로 “화면에서 숨기기”와 “다른 대화를 시작하기”를 구분합니다.

## Streamlit에서 실제 모델에 연결

앱이 위젯 때문에 다시 실행돼도 compiled graph와 현재 thread_id는 같은 세션 안에서 보관해야 합니다. 새 대화 버튼은 UUID로 새 thread_id를 발급합니다. 과거 메시지는 체크포인트에서 한 번만 읽고 새 입력이 들어온 순간에만 모델을 호출합니다. 아래 셀이 앱 파일을 만듭니다. 노트북과 같은 환경의 터미널에서 `python -m streamlit run chapter35_chat.py`를 실행하세요. API 키를 채운 .env가 앱과 같은 폴더에 있어야 합니다.

In [4]:
from pathlib import Path

# 터미널에서 실행할 화면 파일을 현재 폴더에 저장합니다.
app_source = '''from uuid import uuid4
import os
import streamlit as st
from dotenv import load_dotenv
from openai import OpenAI
from langchain_core.messages import AIMessage, HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import START, END, MessagesState, StateGraph

load_dotenv()
st.title("나의 대화 기록")

def respond(state):
    # 노드에 전달된 메시지를 모델의 user/assistant 형식으로 바꿉니다.
    turns = [{"role": "user" if m.type == "human" else "assistant", "content": m.content}
             for m in state["messages"]]
    reply = OpenAI().responses.create(model=os.getenv("OPENAI_MODEL", "gpt-4.1-mini"), store=False, input=turns)
    return {"messages": [AIMessage(content=reply.output_text)]}

if "chat" not in st.session_state:
    plan = StateGraph(MessagesState)
    plan.add_node("respond", respond)
    plan.add_edge(START, "respond")
    plan.add_edge("respond", END)
    st.session_state.chat = plan.compile(checkpointer=InMemorySaver())
if "thread" not in st.session_state:
    st.session_state.thread = str(uuid4())
if st.button("새 대화"):
    st.session_state.thread = str(uuid4())

cfg = {"configurable": {"thread_id": st.session_state.thread}}
chat = st.session_state.chat
for message in chat.get_state(cfg).values.get("messages", []):
    st.chat_message("user" if message.type == "human" else "assistant").write(message.content)
if question := st.chat_input("질문하기"):
    st.chat_message("user").write(question)
    result = chat.invoke({"messages": [HumanMessage(content=question)]}, cfg)
    st.chat_message("assistant").write(result["messages"][-1].content)
'''
Path("chapter35_chat.py").write_text(app_source, encoding="utf-8")
print("chapter35_chat.py")

chapter35_chat.py


이 셀의 출력은 생성한 파일 이름입니다. 화면과 실제 모델 응답은 터미널에서 앱을 실행한 뒤 확인합니다.

## InMemorySaver를 다시 만들면 기록이 어떻게 될까요?

같은 thread_id를 사용해도 체크포인터 자체가 다른 저장 공간이면 앞의 실행을 찾지 못합니다. 화면 재실행 때마다 build_chat을 새로 호출하는 실수를 노트북에서 재현합니다.

In [5]:
second_chat = build_chat(lambda messages: f"현재 발화 수 {len(messages)}")
print("원래 그래프", len(chat.get_state(config("blue")).values["messages"]))
print("새 그래프", second_chat.get_state(config("blue")).values.get("messages", []))

원래 그래프 4
새 그래프 []


첫 그래프에는 blue 대화가 있지만 새 그래프의 사전에는 같은 키가 없습니다. 팀원이 화면을 누를 때마다 이름을 잊는다고 보고하면 모델 지시를 고치기 전에 컴파일한 그래프와 thread_id가 둘 다 보존되는지 점검하세요. 이 수업의 세션 메모리는 서버 재시작을 견디는 저장소가 아닙니다.

## 실습 · 서로 다른 스레드를 바꿔 읽기

별도의 thread_id에 짝의 새 이름을 입력하세요. 같은 ID에 질문한 상태와 한 번도 쓰지 않은 ID의 상태를 비교합니다. 이름이 다른 대화에 나타나면 먼저 어떤 값이 공유되고 있는지 확인하세요.

In [6]:
used_thread, empty_thread = "orange", "unused"
# used_thread에서 send를 두 번 호출하세요. empty_thread는 get_state로만 열어 messages를 비교하세요.

<details><summary>모범 예시와 풀이</summary><pre><code>send(chat, &quot;orange&quot;, &quot;제 이름은 별입니다&quot;)
print(send(chat, &quot;orange&quot;, &quot;첫 말을 기억하나요?&quot;)[&quot;messages&quot;][-1].content)
print(chat.get_state(config(&quot;unused&quot;)).values.get(&quot;messages&quot;, []))</code></pre><p>orange에는 별이 있고 아직 호출하지 않은 unused의 메시지는 빈 목록입니다. get_state는 모델을 호출하지 않습니다.</p></details>

## 실습 · 화면을 새로 그릴 때 중복을 찾아내기

앱에서 이름을 말하고 후속 질문을 보낸 뒤, 새 대화를 시작해 “방금 말한 내 이름은?”이라고 물으세요. 화면에 그려진 발화 개수와 ID별 저장된 발화 개수를 기록합니다. 새 대화가 이름을 우연히 맞히는지보다 실제 전달된 메시지가 무엇인지 확인합니다.

In [7]:
first_window, new_window = "window1", "window2"
# build_chat에 메시지 수를 반환하는 함수를 연결하세요. 같은 질문을 기존 창과 새 창에 보내 수를 비교하세요.

<details><summary>모범 예시와 풀이</summary><pre><code># 노트북에서 화면과 같은 두 요청을 재현합니다.
screen_chat = build_chat(lambda messages: f&quot;모델에 보낼 메시지 수 {len(messages)}&quot;)
send(screen_chat, &quot;window1&quot;, &quot;제 이름은 구름입니다&quot;)
print(send(screen_chat, &quot;window1&quot;, &quot;제 이름은?&quot;)[&quot;messages&quot;][-1].content)
print(send(screen_chat, &quot;window2&quot;, &quot;제 이름은?&quot;)[&quot;messages&quot;][-1].content)</code></pre><p>후속 질문 직전의 모델 입력은 세 개, 새 대화의 첫 모델 입력은 한 개입니다. 답 내용은 모델마다 달라질 수 있어 입력 목록을 증거로 남깁니다.</p></details>

## 전이 과제 · 지난 발화를 두 번 보내는 버그 찾기

화면에 이미 저장한 메시지를 전부 새 객체로 만든 뒤 체크포인터에도 다시 보냈다고 가정합니다. 텍스트가 같아도 서로 다른 메시지 ID를 받으면 누적 규칙은 별개의 발화로 취급합니다. 사용자가 같은 문장을 실제로 두 번 입력하는 경우와 애플리케이션이 복사본을 다시 보내는 경우를 텍스트만으로 구분할 수 있을까요?

같은 문구를 서로 다른 ID와 같은 ID로 각각 전달합니다. 사용자에게 보여 줄 글자 수는 같아도 저장 결과가 어떻게 달라지는지 확인하세요. 새 질문이 한 개 들어올 때 화면이 모델에 넘길 데이터는 무엇이어야 할지 한 줄로 정합니다.

In [8]:
old = [HumanMessage(content="내일 갈게요", id="a")]
# 같은 문장에 새 ID를 붙인 경우와, 바뀐 문장에 기존 ID를 붙인 경우를 add_messages로 비교하세요.

<details><summary>모범 예시와 풀이</summary><pre><code>old = [HumanMessage(content=&quot;내일 갈게요&quot;, id=&quot;a&quot;)]
copy = HumanMessage(content=&quot;내일 갈게요&quot;, id=&quot;b&quot;)
updated = HumanMessage(content=&quot;모레 갈게요&quot;, id=&quot;a&quot;)
print([(m.id, m.content) for m in add_messages(old, [copy])])
print([(m.id, m.content) for m in add_messages(old, [updated])])</code></pre><p>첫 결과는 두 메시지이고 둘째는 a 한 건의 수정입니다. 새 입력을 받으면 새 사용자 발화 하나만 보내고, 옛 대화는 저장된 스레드에서 읽는 구조가 중복을 피하기 쉽습니다. 같은 텍스트가 보인다고 실제 사용자 발화를 함부로 지우면 안 됩니다.</p></details>

다음 장에서는 대화 전체 대신 질문 한 건과 검색 결과를 상태로 들고 다닙니다. [LangGraph 체크포인트](https://docs.langchain.com/oss/python/langgraph/persistence) · [Streamlit 세션 상태](https://docs.streamlit.io/develop/api-reference/caching-and-state/st.session_state)